In [1]:
import numpy as np
import pytz

from Data_query.trino_config import *
from visualisation import *

In [12]:
stop_trino()

Trino service stopping triggered.


In [7]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)

In [8]:
ensure_trino_running(worker_desired_count=workers, big_worker_desired_count=big_workers)
sleep(60)

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [9]:
iceberg_sql(f""" select distinct site_id, ac_capacity_kw, state, min_time, max_time from meta_up23c 
            where is_pv=True and flex_export_detected=False and pf_01 > .98 and min_time < timestamp '2024-01-02'
            limit 2""")

,site_id,ac_capacity_kw,state,min_time,max_time
0,1233585204,5.0,VIC,2024-01-01,2024-06-04 20:55:00
1,158521185,8.0,NSW,2024-01-01,2025-06-06 07:10:00


In [ ]:
# Some sites are excluded because no CS day profile is detected for them. For example, site 1525233041, cs_day is 2024-01-21. But no ts data is detected for that day.

In [10]:
iceberg_exec("drop table if exists split_days")
iceberg_exec(f"""
     create table split_days (
            site_id BIGINT,
            actual_day DATE,
            day_type varchar
        )""")

Executed
Executed


In [ ]:
num_parts = 1
time_bin_interval = "30"  # in minutes


def run_func(args):
    year, month, split_cons, part = args
    # time_filter = f"year = {year} and month = {month}"
    time_filter = f"year = {year}"
    part_filter = f"site_id % {num_parts} = {part}"
    meta_filters = (
        f"is_pv=True and {split_cons} and flex_export_detected=False and {part_filter}"
    )
    # meta_filters = f"is_pv=True and {split_cons} and flex_export_detected=False and site_id in (1669657679,1947677239)"
    df = iceberg_exec(f"""
                    insert into split_days
                    with data as 
                        (select 
                            site_id, t_stamp,  sum(power*circuit_polarity/S_99)/1000 as P_kw_norm,
                            sum(energy_reactive*circuit_polarity/S_99)/1000*12 as Q_kvar_norm, 
                            avg(voltage) as V
                        from ts join 
                            (select site_id, circuit_id, circuit_polarity, S_99  from meta_up23c where {meta_filters})
                            as m on ts.circuit_id = m.circuit_id
                        where {time_filter} and {part_filter} and is_pv=True and voltage >= 0 and voltage <= 300 and {split_cons}
                        group by site_id, t_stamp
                            ),
                    bom10min as (
                        select 
                            distinct time, b.latitude, b.longitude, surface_global_irradiance as GHI, cloud_type
                        from bom_nci.solar as b
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) as m 
                            on b.latitude = m.n_lat and b.longitude = m.n_long
                        where {time_filter} 
                    ),
                    bom5min as ((select time as time_5min, latitude, longitude, GHI, cloud_type
                    from bom10min) 
                    union all
                    (select date_add('minute', 5, time) as time_5min, latitude, longitude, GHI, cloud_type
                    FROM bom10min ORDER BY time_5min)),
                    daily_cloud AS (
                        SELECT
                            latitude, longitude,
                            date_trunc('day', time + interval '10' hour)   AS day,
                            date_trunc('month', time + interval '10' hour) AS month,
                            sum(cloud_type) AS cloud_sum, 
                            max(GHI) AS max_GHI
                        FROM bom10min
                        GROUP BY
                            1, 2, 3, 4
                    ),
                    clear_sky AS (
                            SELECT day, latitude, longitude
                            FROM (SELECT day, latitude, longitude, cloud_sum, max_GHI, row_number() OVER (
                                                                    PARTITION BY month, latitude, longitude
                                                                    ORDER BY cloud_sum ASC, day ASC
                                                                    ) AS rn
                                FROM daily_cloud 
                            )
                            WHERE rn < 4 and cloud_sum < 60 and max_GHI > 200
                        ),
                    daily_site_days AS (
                            SELECT 
                                n_lat,
                                n_long,
                                date_trunc('day', t_stamp + interval '10' hour) AS day
                            FROM data d join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                            on d.site_id = m.site_id
                            group by n_lat, n_long, date_trunc('day', t_stamp + interval '10' hour) 
                        ),
                    nearest_clear_sky_day AS (
                        SELECT
                            dy.n_lat,
                            dy.n_long,
                            dy.day AS actual_day,
                            c.day AS clear_sky_day,
                            row_number() OVER (
                                PARTITION BY dy.n_lat, dy.n_long, dy.day
                                ORDER BY abs(date_diff('day', dy.day, c.day)), date_diff('day', c.day, dy.day)
                            ) AS rn
                        FROM daily_site_days dy JOIN clear_sky c
                            ON dy.n_lat = c.latitude AND dy.n_long = c.longitude
                    ),
                    nearest_clear_sky AS (
                            SELECT
                                n_lat,
                                n_long,
                                actual_day,
                                clear_sky_day
                            FROM nearest_clear_sky_day
                            WHERE rn = 1
                        ),
                    nearest_cs_days AS (
                        SELECT
                            DISTINCT site_id, clear_sky_day AS cs_day 
                        FROM nearest_clear_sky n JOIN (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                            ON n.n_lat = m.n_lat AND n.n_long = m.n_long
                    ),
                    base AS (
                            SELECT
                                d.*,
                                lag(t_stamp) OVER (
                                    PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                    ORDER BY t_stamp
                                ) AS prev_ts
                            FROM data d
                    ),
                    gaps AS (
                        SELECT *,
                            CASE
                                WHEN prev_ts IS NULL THEN 0
                                WHEN t_stamp - prev_ts > interval '30' minute THEN 1
                                ELSE 0
                            END AS gap_start
                        FROM base
                    ),
                    segments AS (
                        SELECT *,
                            sum(gap_start) OVER (
                                PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                ORDER BY t_stamp
                                ROWS UNBOUNDED PRECEDING
                            ) AS segment_id
                        FROM gaps
                    ),
                    nearest_cs_profiles AS (
                        SELECT
                            s.site_id,
                            date_trunc('day', s.t_stamp + interval '10' hour) AS cs_day,
                            (s.t_stamp + interval '10' hour) - date_trunc('day', s.t_stamp + interval '10' hour) AS cs_tod,
                            approx_percentile(P_kw_norm, 0.6) OVER (
                                    PARTITION BY s.site_id, date_trunc('day', s.t_stamp + interval '10' hour), segment_id
                                        ORDER BY t_stamp
                                        ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
                                    ) AS P_kw_norm_cs,
                            approx_percentile(GHI, 0.6) OVER (
                            PARTITION BY s.site_id, date_trunc('day', s.t_stamp + interval '10' hour), segment_id
                                ORDER BY t_stamp
                                ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
                            ) AS GHI_cs,
                            cloud_type as cloud_type_cs
                        FROM segments s join nearest_cs_days n on s.site_id = n.site_id and 
                            date_trunc('day', s.t_stamp + interval '10' hour) = n.cs_day
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m on s.site_id = m.site_id
                            join bom5min b on m.n_lat = b.latitude and m.n_long = b.longitude and b.time_5min = s.t_stamp
                    ),
                    strcutured_data AS (
                        SELECT
                            d.site_id,
                            d.t_stamp,
                            date_trunc('day', d.t_stamp + interval '10' hour) AS actual_day,
                            (d.t_stamp + interval '10' hour) - date_trunc('day', d.t_stamp + interval '10' hour) AS actual_tod,
                            d.V,
                            d.Q_kvar_norm,
                            d.P_kw_norm, 
                            sqrt(pow(d.Q_kvar_norm, 2) + pow(d.P_kw_norm, 2)) AS S_norm,
                            GHI,
                            cloud_type,
                            ncs.cs_day,
                            ncs.cs_tod,
                            ncs.P_kw_norm_cs,
                            ncs.GHI_cs, 
                            ncs.cloud_type_cs
                        FROM data d 
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                                ON d.site_id = m.site_id 
                            join nearest_cs_profiles ncs 
                                on d.site_id = ncs.site_id and ncs.cs_tod = (d.t_stamp + interval '10' hour) - date_trunc('day', d.t_stamp + interval '10' hour)
                            join nearest_clear_sky n on 
                                n.n_lat = m.n_lat AND n.n_long = m.n_long and n.actual_day = date_trunc('day', d.t_stamp + interval '10' hour)
                                and n.clear_sky_day = ncs.cs_day
                            join bom5min b on m.n_lat = b.latitude and m.n_long = b.longitude and b.time_5min = d.t_stamp
                        Where abs(date_diff('day', ncs.cs_day, n.actual_day)) < 45
                            order by d.site_id, d.t_stamp),
                    train_val_data AS (
                        SELECT
                            site_id,
                            actual_day,
                            t_stamp,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % {time_bin_interval})
                                AS TIME) AS tod_bin,
                            GHI AS x,
                            P_kw_norm/ NULLIF(P_kw_norm_cs, 0.0) AS y
                        FROM strcutured_data
                        WHERE P_kw_norm_cs > 0.2 AND GHI > 50 and P_kw_norm > 0.05
                            and V <= 253 and (P_kw_norm >= 1 or S_norm < 1.001)
                    ),
                    random_days AS (
                        SELECT 
                            site_id,
                            actual_day,
                            row_number() OVER (PARTITION BY site_id ORDER BY random()) AS rn,
                            count(*) OVER (PARTITION BY site_id) AS total_days
                        FROM (SELECT DISTINCT site_id, actual_day FROM train_val_data) AS distinct_days
                    ),
                    split_days AS (
                        SELECT
                            site_id,
                            actual_day,
                            CASE
                                WHEN rn <= 0.8 * total_days THEN 'train'
                                ELSE 'val'
                            END AS day_type
                        FROM random_days
                    )
                    select * from split_days
                    
                
    """)

    # print(f"Completed {time_filter}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}, part {part}, count: {df['site_id'].nunique()}")
    # sleep(3)
    return df


tasks = [
    (year, month, split_cons, part)
    for year in (2024,)
    for month in range(1, 2)
    for split_cons in [f"system.bucket(postcode, 16) = {i}" for i in range(0, 16)]
    for part in range(0, num_parts)
]
#   for split_cons in ['system.bucket(postcode, 16) > -1'] ]

try:
    res = trino_parallel(run_func, tasks, num_workers=num_workers)
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
res

Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
